In [1]:
import mne
import os
import pandas as pd
import sys
sys.path.append("../Preprocessing")
from preprocessingPipeline import simplePipeline
from convertTUHtoBIDS import CHANNELS_TO_KEEP

In [77]:
def _rename_channels(raw):
	"""
	Renames the EEG channels using mne conventions and sets their type to 'eeg'.

	See https://isip.piconepress.com/publications/reports/2020/tuh_eeg/electrodes/
	"""
	# remove ref suffix and prefix:
	# TODO: replace with removesuffix and removeprefix when 3.8 is dropped
	mapping_strip = {
		c: c.replace("-REF", "").replace("-LE", "").replace("EEG ", "")
		for c in raw.ch_names
	}
	raw.rename_channels(mapping_strip)

	montage1005 = mne.channels.make_standard_montage("standard_1005")

	# Changed this line
	mapping_eeg_names = {
		c.upper(): c for c in montage1005.ch_names if c.upper() in raw.ch_names
	}

	# Set channels whose type could not be inferred (defaulted to "eeg") to "misc":
	non_eeg_names = [c for c in raw.ch_names if c not in mapping_eeg_names]
	if non_eeg_names:
		non_eeg_types = raw.get_channel_types(picks=non_eeg_names)
		mapping_non_eeg_types = {
			c: "misc" for c, t in zip(non_eeg_names, non_eeg_types) if t == "eeg"
		}
		if mapping_non_eeg_types:
			raw.set_channel_types(mapping_non_eeg_types)

	if mapping_eeg_names:
		# Set 1005 channels type to "eeg":
		raw.set_channel_types(
			{c: "eeg" for c in mapping_eeg_names}, on_unit_change="ignore"
		)
		# Fix capitalized EEG channel names:
		raw.rename_channels(mapping_eeg_names)

In [88]:
epochs = process_session('/home/azureuser/mycontainer/TUEV/v2.0.1/edf/eval/000/')
print(epochs.annotations)

Extracting EDF parameters from /home/azureuser/mycontainer/TUEV/v2.0.1/edf/eval/000/bckg_000_a_.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 306499  =      0.000 ...  1225.996 secs...


/tmp/ipykernel_4137/403806535.py:30: RuntimeWarning: The unit for channel(s) 26, 27, 28, 29, 30, EKG1, EMG, LOC, PHOTIC, ROC, T1, T2 has changed from V to NA.
  raw.set_channel_types(mapping_non_eeg_types)


Not setting metadata
20 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 20 events and 15000 original time points ...
0 bad epochs dropped
<Annotations | 633 segments, channel-specific: 4 (22), 5 (114), 6 (497)>


In [89]:
print(epochs.get_data().shape)

(18, 19, 7680)


In [10]:
raw_epoch = mne.read_epochs("/home/azureuser/mycontainer/TUEV_Processed/train/aaaaaaar_00000001_epo.fif", preload=True, verbose=False)
annotations_per_epoch = raw_epoch.get_annotations_per_epoch()
raw_data = raw_epoch.get_data(copy=True)
print(raw_data.shape)
len(annotations_per_epoch)
for idx, annotations in enumerate(annotations_per_epoch):
	print(idx, annotations)
	descriptions = [int(annotation[2]) for annotation in annotations]
	if len(set(descriptions)) > 1:
		print(set(descriptions))

(26, 19, 7680)
0 [(4.699999809265137, 1.0, '5'), (4.699999809265137, 1.0, '5'), (4.699999809265137, 1.0, '5'), (4.800000190734863, 1.0, '6'), (4.800000190734863, 1.0, '6'), (4.800000190734863, 1.0, '5'), (4.800000190734863, 1.0, '5'), (4.800000190734863, 1.0, '6'), (4.800000190734863, 1.0, '6'), (4.800000190734863, 1.0, '6'), (4.800000190734863, 1.0, '6'), (4.800000190734863, 1.0, '6'), (4.800000190734863, 1.0, '6'), (4.800000190734863, 1.0, '6'), (4.800000190734863, 1.0, '6'), (4.800000190734863, 1.0, '6'), (4.900000095367432, 1.0, '5'), (5.699999809265137, 1.0, '5'), (5.699999809265137, 1.0, '5'), (5.699999809265137, 1.0, '5'), (5.800000190734863, 1.0, '6'), (5.800000190734863, 1.0, '6'), (5.800000190734863, 1.0, '5'), (5.800000190734863, 1.0, '5'), (5.800000190734863, 1.0, '6'), (5.800000190734863, 1.0, '6'), (5.800000190734863, 1.0, '6'), (5.800000190734863, 1.0, '6'), (5.800000190734863, 1.0, '6'), (5.800000190734863, 1.0, '6'), (5.800000190734863, 1.0, '6'), (5.800000190734863, 1